# Case - Engenharia de Dados com PySpark e Databricks

## Objetivo

Este projeto demonstra a construção de um pipeline de dados utilizando
PySpark e Delta Lake seguindo a arquitetura Medalhão (Raw, Bronze, Silver e Gold).

Ao final será disponibilizado um modelo analítico (Star Schema) e uma
consulta que gera um JSON que traga como resultado a quantidade e valor total de venda por País e Linha de Produto, somente das ordens de venda com status "Shipped".

## Arquitetura proposta para resolução do case.

| Camada     | Objetivo                                             |
|------------|------------------------------------------------------|
| 📄 Raw     | Armazenar o arquivo original sem alterações.         |
| 🥉 Bronze  | Aplicar padronizações técnicas e preservar os dados. |
| 🥈 Silver  | Aplicar regras de negócio, limpeza e enriquecimento. |
| 🥇 Gold    | Disponibilizar modelo dimensional (Star Schema).     |
| 📊 Consumo | Consultas analíticas e geração do JSON final.        |

# Configuração do projeto

In [0]:
# Importação de bibliotecas
import json

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import *

from pyspark.sql.window import Window

from delta.tables import DeltaTable

### Criação do catálogo, esquema e volumes necessários para execução do projeto

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS case;

CREATE SCHEMA IF NOT EXISTS case.sales;

CREATE VOLUME IF NOT EXISTS case.sales.raw;

CREATE VOLUME IF NOT EXISTS case.sales.bronze;

CREATE VOLUME IF NOT EXISTS case.sales.silver;

CREATE VOLUME IF NOT EXISTS case.sales.gold;

In [0]:
# Definições de caminhos para as tabelas
SOURCE_FILE = "/Workspace/Sales/base_case_engenheiro_dados.csv"
RAW_PATH = "/Volumes/case/sales/raw"
BRONZE_PATH = "/Volumes/case/sales/bronze"
SILVER_PATH = "/Volumes/case/sales/silver"
GOLD_PATH = "/Volumes/case/sales/gold"

#Inicialização com a visualização dos dados

In [0]:
# Leitura do arquivo como texto puro para inspecionar possíveis desvios de estrutura.

df_raw_text = spark.read.text(SOURCE_FILE)
df_raw_text.show(truncate=False)

Na leitura do arquivo acima é possível identificar algumas inconsistências como: datas em formatos diferentes, coluna de QuantityOrdered com valores decimais (exemplo orderID O0005), coluna de total vendido (TotalSales) em formatos diferentes com separados de decimais com vírgula ao invés de ponto (exemplo na linha com orderID O0003).

Como a coluna TotalSales é calculada, posteriormente será recriada a coluna realizando o cálculo do total de vendas garantindo qualidade e confiabilidade dos dados.

É possível também identificar problemas nas descrição dos países, como Brasil e Brazil, esses dados serão normalizados.


In [0]:
# Salvando o arquivo na camada raw em formato Delta
df_raw_text.write.mode("overwrite").format("delta").save(RAW_PATH)

# Camada Bronze

In [0]:
df = spark.read.format("delta").load(RAW_PATH)
display(df)


In [0]:
# criação de uma função para tratamento do arquivo csv puro, ignorando a coluna TotalSales do csv e convertendo a coluna QuantityOrdered para int para remoção dos decimais

def create_df_bronze(
    df: DataFrame,
    coluna: str = "value",
    separador: str = ","
) -> DataFrame:

    return (
        df
        .filter(~F.col(coluna).startswith("OrderID"))
        .select(F.split(F.col(coluna), separador).alias("cols"))
        .select(
            F.col("cols")[0].alias("OrderID"),
            F.col("cols")[1].alias("OrderDate"),
            F.col("cols")[2].alias("Status"),
            F.col("cols")[3].alias("CustomerID"),
            F.col("cols")[4].alias("Country"),
            F.col("cols")[5].alias("ProductID"),
            F.col("cols")[6].alias("ProductLine"),
            F.col("cols")[7].cast("double").cast("int").alias("QuantityOrdered"),
            F.col("cols")[8].cast("double").alias("Price")
        )
    )


In [0]:
# chamada da função para criação do dataframe bronze
df_bronze = create_df_bronze(df)
df_bronze.show(5)

In [0]:
# Salvar o dataframe tratato na camada Silver
df_bronze.write.mode("overwrite").format("delta").save(BRONZE_PATH)

# Camada Silver


In [0]:
# definição das colunas que precisam de correção de espaçamento, remoção de espaços vazios (Trim)
STRING_COLUMNS = [
    "OrderID",
    "OrderDate",
    "Status",
    "CustomerID",
    "Country",
    "ProductID",
    "ProductLine"
]

In [0]:
def load_bronze(path: str) -> DataFrame:
    """Realiza a leitura da camada Bronze."""
    return spark.read.format("delta").load(path)

In [0]:
def trim_string_columns(df: DataFrame) -> DataFrame:
    """Remove espaços vazios nas colunas."""
    return df.select(
        *[
            F.trim(F.col(c)).alias(c)
            for c in STRING_COLUMNS
        ],
        "QuantityOrdered",
        "Price"
    )

In [0]:
def cast_columns(df: DataFrame) -> DataFrame:
    """Conversão de tipagem das colunas"""
    return (
        df
        .withColumn(
            "QuantityOrdered",
            F.col("QuantityOrdered").cast("int")
        )
        .withColumn(
            "Price",
            F.col("Price").cast("double")
        )
    )

In [0]:
def add_total_sales(df: DataFrame) -> DataFrame:
    """Criação da coluna TotalSales."""
    return df.withColumn(
        "TotalSales",
        F.round(
            F.col("QuantityOrdered") * F.col("Price"),
            2
        )
    )

In [0]:
df_bronze = load_bronze(BRONZE_PATH)

df_silver = trim_string_columns(df_bronze)

df_silver = cast_columns(df_silver)

df_silver = add_total_sales(df_silver)

display(df_silver.limit(20))

In [0]:
# Verificação dos nomes dos países desnormalizados
display(
    df_silver
        .select("Country")
        .distinct()
        .orderBy("Country")
)

In [0]:
# existem países com nomes desnormalizados sendo: Brasil/Brazil, France/França, Alemanha/Germany, criação da função para normalização

def normalize_country(df: DataFrame) -> DataFrame:
    """
    Padroniza os valores da coluna Country.
    """

    return (
        df.withColumn(
            "Country",
            F.when(F.lower(F.col("Country")).isin("brasil", "brazil"), "Brazil")
             .when(F.lower(F.col("Country")).isin("frança", "franca", "france"), "France")
             .when(F.lower(F.col("Country")).isin("alemanha", "germany"), "Germany")
             .otherwise(F.col("Country"))
        )
    )

In [0]:
# execução da função de normalização
df_silver = normalize_country(df_silver)
display(df_silver.select("Country").distinct())

In [0]:
# Verificação da existência de registros duplicados de venda.

duplicates = (
    df_silver
        .groupBy(*df_silver.columns)
        .count()
        .filter(F.col("count") > 1)
)

if duplicates.limit(1).count() == 0:
    print("Nenhum registro duplicado encontrado.")
else:
    print("Registros duplicados encontrados:")
    display(duplicates)

    # Remove registros completamente duplicados
    df_silver = df_silver.dropDuplicates()
    print("Registros duplicados removidos")

In [0]:
# Verifica os diferentes formatos encontrados na coluna OrderDate
(
    df_silver
        .select("OrderDate")
        .distinct()
        .orderBy("OrderDate")
        .show(100, truncate=False)
)

In [0]:
# Cria uma máscara para identificar padrões de datas.

df_patterns = (
    df_silver
        .withColumn(
            "DatePattern",
            F.regexp_replace(
                F.regexp_replace(F.col("OrderDate"), r"\d", "X"),
                r"[A-Za-z]", "L"
            )
        )
)

df_analysis = (
    df_patterns
        .groupBy("DatePattern")
        .agg(
            F.count("*").alias("RecordCount"),
            F.min("OrderDate").alias("ExampleValue")
        )
        .orderBy(F.desc("RecordCount"))
)

display(df_analysis)

In [0]:
# definição de formatos de datas que serão usados para normalizar a coluna OrderDate
DATE_FORMATS = [
    "yyyy-MM-dd",
    "yyyy/MM/dd",
    "dd-MM-yyyy",
    "dd/MM/yyyy",
    "MM/dd/yyyy",
    "yyyy/MM-dd",
    "yyyy-MMM-dd",
    "yyyy-MM-dd HH:mm:ss",
    "yyyy/MM/dd HH:mm:ss"
]

# função para normalizar as datas
def normalize_date_column(
    df: DataFrame,
    column_name: str
) -> DataFrame:
    """
    Normaliza a coluna OrderDate para o tipo Date,
    tentando converter os formatos conhecidos.
    """

    return (
        df.withColumn(
            "OrderDate",
            F.coalesce(
                *[
                    F.to_date(
                        F.try_to_timestamp(
                            F.col("OrderDate"),
                            F.lit(fmt)
                        )
                    )
                    for fmt in DATE_FORMATS
                ]
            )
        )
    )

In [0]:
# Execução da função
df_silver = normalize_date_column(df_silver, "OrderDate")

In [0]:
# validação da conversão das datas
invalid_dates = (
    df_silver
        .filter(F.col("OrderDate").isNull())
        .count()
)

print(f"Datas não convertidas: {invalid_dates}")

## Etapa de Data Quality na camada Silver

In [0]:
# verificação de integridade para verificar se existem dados inconsistentes na coluna QuantityOrdered, exemplo valores menores ou iguais a 0
invalid_quantity = (
    df_silver
        .filter(F.col("QuantityOrdered") <= 0)
)

invalid_count = invalid_quantity.count()

if invalid_count > 0:
    print(f"{invalid_count} registro(s) com QuantityOrdered <= 0 encontrados.")
    print("Removendo registros inválidos...")

    df_silver = (
        df_silver
            .filter(F.col("QuantityOrdered") > 0)
    )
else:
    print("Nenhum registro inválido encontrado.")

In [0]:
# verificação de integridade para verificar se existem dados inconsistentes na coluna Price, exemplo valores menores ou iguais a 0
invalid_price = (
    df_silver
        .filter(F.col("Price") <= 0)
)

invalid_count = invalid_price.count()

if invalid_count > 0:
    print(f"{invalid_count} registro(s) com Price <= 0 encontrados.")
    print("Removendo registros inválidos...")

    df_silver = (
        df_silver
            .filter(F.col("Price") > 0)
    )
else:
    print("Nenhum registro inválido encontrado.")

In [0]:
# verificação de integridade para verificar se existem dados inconsistentes na coluna Status
invalid_status = (
    df_silver
        .filter(F.col("Status").isNull())
)

invalid_count = invalid_status.count()

if invalid_count > 0:
    print(f"{invalid_count} registro(s) com Status null encontrados.")
    print("Removendo registros inválidos...")

    df_silver = (
        df_silver
            .filter(F.col("Status").isNotNull())
    )
else:
    print("Nenhum registro inválido encontrado.")

In [0]:
# verificação de integridade para verificar se existem dados inconsistentes na coluna country
invalid_country = (
    df_silver
        .filter(F.col("Country").isNull())
)

invalid_count = invalid_country.count()

if invalid_count > 0:
    print(f"{invalid_count} registro(s) com Country null encontrados.")
    print("Removendo registros inválidos...")

    df_silver = (
        df_silver
            .filter(F.col("Country").isNotNull())
    )
else:
    print("Nenhum registro inválido encontrado.")

In [0]:
# alteração da nomenclatura das colunas para um padrão mais semântico
df_silver = (
    df_silver.select(
        F.col("OrderID").alias("order_id"),
        F.col("OrderDate").alias("order_date"),
        F.col("Status").alias("status"),
        F.col("CustomerID").alias("customer_id"),
        F.col("Country").alias("country"),
        F.col("ProductID").alias("product_id"),
        F.col("ProductLine").alias("product_line"),
        F.col("QuantityOrdered").alias("quantity_ordered"),
        F.col("Price").alias("unit_price"),
        F.col("TotalSales").alias("total_sales")
    )
)
display(df_silver)


In [0]:
# salvando o arquivo final na pasta Silver

df_silver.write.mode("overwrite").format("delta").save(SILVER_PATH)

# Camada Gold

## Criação do modelo analítico

In [0]:
# leitura do arquivo salvo na camada Silver após os tratamentos
df_silver = spark.read.format("delta").load(SILVER_PATH)

In [0]:
# Tabelas dimensionais
DIM_DATE_PATH = f"{GOLD_PATH}/dim_date"
DIM_CUSTOMER_PATH = f"{GOLD_PATH}/dim_customer"
DIM_PRODUCT_PATH = f"{GOLD_PATH}/dim_product"
DIM_STATUS_PATH = f"{GOLD_PATH}/dim_status"

# Tabela fato
FACT_SALES_PATH = f"{GOLD_PATH}/fact_sales"

In [0]:
# Criação da dimensão de datas

dim_date = (
    df_silver
        .select("order_date")
        .filter(F.col("order_date").isNotNull())
        .distinct()
        .select(
            # Chave da dimensão no formato YYYYMMDD, facilitando os joins com a tabela fato.
            F.date_format("order_date", "yyyyMMdd").cast("int").alias("date_key"),

            F.col("order_date").alias("full_date"),

            F.year("order_date").alias("year"),
            F.quarter("order_date").alias("quarter"),
            F.month("order_date").alias("month"),
            F.date_format("order_date", "MMMM").alias("month_name"),

            F.dayofmonth("order_date").alias("day"),
            F.dayofweek("order_date").alias("day_of_week"),
            F.weekofyear("order_date").alias("week_of_year"),

            F.when(
                F.dayofweek("order_date").isin(1, 7),
                True
            ).otherwise(False).alias("is_weekend")
        )
        .orderBy("full_date")
)

# Visualização da dimensão criada
display(dim_date)

# Persistência da dimensão na camada Gold
(
    dim_date.write
        .format("delta")
        .mode("overwrite")
        .save(DIM_DATE_PATH)
)

In [0]:
# Criação da dimensão de clientes

dim_customer = (
    df_silver
        .select("customer_id", "country")
        .filter(F.col("customer_id").isNotNull())
        .dropDuplicates()
        .orderBy("customer_id")
)

# Visualização da dimensão criada
display(dim_customer)

# Persistência da dimensão na camada Gold
(
    dim_customer.write
        .format("delta")
        .mode("overwrite")
        .save(DIM_CUSTOMER_PATH)
)

In [0]:
# Criação da dimensão de produtos

dim_product = (
    df_silver
        .select(
            "product_id",
            "product_line"
        )
        .filter(F.col("product_id").isNotNull())
        .dropDuplicates()
        .orderBy("product_id")
)

# Visualização da dimensão criada
display(dim_product)

# Persistência da dimensão na camada Gold
(
    dim_product.write
        .format("delta")
        .mode("overwrite")
        .save(DIM_PRODUCT_PATH)
)

In [0]:
# Criação da dimensão de status

dim_status = (
    df_silver
        .select("status")
        .filter(F.col("status").isNotNull())
        .dropDuplicates()
        .orderBy("status")
)

# Visualização da dimensão criada
display(dim_status)

# Persistência da dimensão na camada Gold
(
    dim_status.write
        .format("delta")
        .mode("overwrite")
        .save(DIM_STATUS_PATH)
)

In [0]:
# Criação da tabela fato contendo as chaves das dimensões e as métricas de negócio para análises na camada Gold.

fact_sales = (
    df_silver
        .withColumn(
            "date_key",
            F.date_format("order_date", "yyyyMMdd").cast("int")
        )
        .select(
            "order_id",
            "date_key",
            "customer_id",
            "product_id",
            "status",
            "quantity_ordered",
            "unit_price",
            "total_sales"
        )
)

# Visualização da tabela fato
display(fact_sales)

# Persistência da tabela fato na camada Gold
(
    fact_sales.write
        .format("delta")
        .mode("overwrite")
        .save(FACT_SALES_PATH)
)

In [0]:
# Registro das tabelas temporárias para consultas spark SQL

fact_sales.createOrReplaceTempView("fact_sales")
dim_customer.createOrReplaceTempView("dim_customer")
dim_product.createOrReplaceTempView("dim_product")
dim_status.createOrReplaceTempView("dim_status")

In [0]:
def generate_sales_json_sql(
    fact_sales: DataFrame,
    dim_customer: DataFrame,
    dim_product: DataFrame,
    dim_status: DataFrame
) -> str:
    """
    Gera um JSON contendo a quantidade e o valor total das vendas
    por país e linha de produto, considerando apenas pedidos
    com status 'Shipped'.
    """

    query = """

    WITH base_sales AS (

        SELECT

            c.country,
            p.product_line,

            SUM(f.quantity_ordered) AS quantity_ordered,
            ROUND(SUM(f.total_sales), 2) AS total_sales

        FROM fact_sales f

        INNER JOIN dim_customer c
            ON f.customer_id = c.customer_id

        INNER JOIN dim_product p
            ON f.product_id = p.product_id

        INNER JOIN dim_status s
            ON f.status = s.status

        WHERE s.status = 'Shipped'

        GROUP BY
            c.country,
            p.product_line

    ),

    country_group AS (

        SELECT

            country,

            SORT_ARRAY(

                COLLECT_LIST(

                    NAMED_STRUCT(

                        'ProductLine', product_line,
                        'QuantityOrdered', CAST(quantity_ordered AS STRING),
                        'Sales', CAST(total_sales AS STRING)

                    )

                )

            ) AS sales

        FROM base_sales

        GROUP BY country

    )

    SELECT

        TO_JSON(

            NAMED_STRUCT(

                'data',

                COLLECT_LIST(

                    NAMED_STRUCT(

                        'Country', country,
                        'Sales', sales

                    )

                )

            )

        ) AS json_result

    FROM country_group

    """

    # Executa a consulta SQL
    df_result = spark.sql(query)

    # Recupera o json gerado pelo Spark
    json_result = df_result.first()["json_result"]

    # Retorna o json formatado
    return json.dumps(
        json.loads(json_result),
        indent=2,
        ensure_ascii=False
    )

In [0]:
print(generate_sales_json_sql(
    fact_sales,
    dim_customer,
    dim_product,
    dim_status
))

In [0]:
def generate_sales_json(
    fact_sales: DataFrame,
    dim_customer: DataFrame,
    dim_product: DataFrame,
    dim_status: DataFrame
) -> str:
    """
    Gera um JSON contendo a quantidade e o valor total das vendas
    por país e linha de produto, considerando apenas pedidos
    com status 'Shipped'.
    """

    # Consolidação das dimensões e agregação das vendas
    sales_df = (
        fact_sales.alias("f")
            .join(
                dim_customer.alias("c"),
                F.col("f.customer_id") == F.col("c.customer_id"),
                "inner"
            )
            .join(
                dim_product.alias("p"),
                F.col("f.product_id") == F.col("p.product_id"),
                "inner"
            )
            .join(
                dim_status.alias("s"),
                F.col("f.status") == F.col("s.status"),
                "inner"
            )
            .filter(F.col("s.status") == "Shipped")
            .groupBy(
                F.col("c.country"),
                F.col("p.product_line")
            )
            .agg(
                F.sum("f.quantity_ordered").alias("quantity_ordered"),
                F.round(F.sum("f.total_sales"), 2).alias("total_sales")
            )
    )

    # Estrutura os produtos agrupados por país
    country_sales = (
        sales_df
            .groupBy(
                F.col("country").alias("Country")
            )
            .agg(
                F.collect_list(
                    F.struct(
                        F.col("product_line").alias("ProductLine"),
                        F.col("quantity_ordered").cast("string").alias("QuantityOrdered"),
                        F.format_string("%.2f", F.col("total_sales")).alias("Sales")
                    )
                ).alias("Sales")
            )
    )

    # Monta o json final
    json_result = (
        country_sales
            .agg(
                F.to_json(
                    F.struct(
                        F.collect_list(
                            F.struct(
                                "Country",
                                "Sales"
                            )
                        ).alias("data")
                    )
                ).alias("json_result")
            )
    )

    # Recupera o json gerado pelo spark
    raw_json = json_result.first()["json_result"]

    # Retorna o json formatado
    return json.dumps(
        json.loads(raw_json),
        indent=2,
        ensure_ascii=False
    )

In [0]:
# Executa a função e gera o json
print(generate_sales_json(
    fact_sales,
    dim_customer,
    dim_product,
    dim_status
))